In [ ]:
import lumicks.pylake as lk
import os
import matplotlib.pyplot as plt
import numpy as np

# Use widget if you're using Jupyter lab or notebook
# %matplotlib widget

# Use notebook if you're in nbclassic
%matplotlib notebook

## Download the kymograph data

In [ ]:
folder_path = r'/path/' # change path
h5_files = [f for f in os.listdir(folder_path) if f.endswith('.h5')]
for file in h5_files:
    print(file)

In [ ]:
file_name = 'file.h5' # pick file
file_path = os.path.join(folder_path, file_name)
file = lk.File(file_path)

## Plotting the kymograph

Let’s load our Bluelake data and have a look at what the kymograph looks like. We can easily grab the kymo by calling `popitem()` on the list of kymos, which returns the first kymograph:

In [ ]:
_, kymo = file.kymos.popitem()

In this experiment, force was measured alongside the kymograph. Let’s plot them together to get a feel for what the data looks like:

In [ ]:
plt.figure(figsize=(7, 4))

# Plot the kymograph
ax1 = plt.subplot(2, 1, 1)

adjustment = lk.ColorAdjustment(0, 1)

# We use aspect="auto" because otherwise the kymograph would be very long and thin
kymo.plot("red", adjustment=adjustment, aspect="auto")

# Plot the force
ax2 = plt.subplot(2, 1, 2, sharex = ax1)
plt.xlim(ax1.get_xlim())
file["Force LF"]["Force 1x"].plot()
plt.tight_layout()


plt.show()

In [ ]:
#select region and bin

kymo_reg= kymo["0s":"150s"]
kymo_reg = kymo_reg.crop_by_distance(3, 17)
kymo_reg_ds = kymo_reg.downsampled_by(1)

plt.figure()
kymo_reg_ds.plot("red", aspect="auto", adjustment=adjustment)
plt.show()

## Performing the kymotracking

Now that we’ve loaded some data, we can begin tracking lines on it. For this, we open the widget. This will provide us with a view of the kymograph and some dials to tune in algorithm settings. We open with a custom `axis_aspect_ratio` which determines our field of view. This input is not necessary, but provides a better view of our data.

We will be using the greedy algorithm. For more information on how it works, please refer to the [Pylake kymotracking tutorial](https://lumicks-pylake.readthedocs.io/en/stable/examples/cas9_kymotracking/../../tutorial/kymotracking.html#track-greedy). The `threshold` should typically be chosen somewhere between the expected baseline photon count and the photon count of a true track (note that you can see the local photon count between square brackets while hovering over the kymograph). The `track width` should roughly be set to the expected spot size (in the spatial dimension) of a track. The `window` should be chosen such that small gaps in a track can be overcome, but not so large that spurious points may be strung together as a track. `Sigma` controls how much the location can fluctuate from one time point to the next, while the `min length` determines how many peak points should be in a track for it to be considered a valid track. The optional `adjacency_filter` removes detections that have no detections in their neighboring frame (prior to tracking) which can cut down on noise.

Holding down the left mouse button and dragging pans the view, while the right mouse button allows us to drag a region where we should perform tracking. Any track which overlaps with the selected area will be removed before tracking new ones.

The icon with the little square can be used to toggle zoom mode, which will allow you to zoom in one subsection of the kymograph. Clicking it again brings us back out of zoom mode. You can zoom out again by clicking the home button. Quite often, it is beneficial to find some adequate settings for track all, and then fine-tune the results using the manual rectangle selection. It’s not mandatory to use the same settings throughout the kymograph. For example, if you see a particular event where two tracks are disconnected but should be connected, temporarily increase the window size and just drag a rectangle over that particular track while having the option `Track` enabled.

Now, let’s do some tracking. There are two ways to approach this analysis. The first is to just use the rectangle selection, which can be quite time intensive. Alternatively, you can use `Track All` to simply track the entire kymograph, and then remove spurious detections by hand. This can be good to get a feel for the parameters as well. If we select the `Remove Tracks` mode we will start removing tracks without grabbing new ones. This functionality can be used to remove spurious detections.

Finally, if you wish to connect two tracks in the kymograph manually, you can switch to the `Connect Tracks` mode. In this mode you can click a point in one track with the right mouse button and connect it to another by dragging to a point in the track you wish to connect it to.

Note that in this data for example, there are some regions where fluorescence starts building up on the surface of the bead. This binding should be omitted from the analysis:

In [ ]:
kymowidget = lk.KymoWidgetGreedy(
    kymo_reg_ds,
    "red",
    adjustment = lk.ColorAdjustment(0, 4),
    axis_aspect_ratio=1,
    
    pixel_threshold=0.5,
    sigma=0.4,
    window=6,
    min_length=8,
    vmax=8,
    #adjacency_filter=True,
    #cmap="viridis",
    cmap="grey",
    correct_origin=True
)
from IPython.display import display, HTML

plt.grid(False)

# Set a larger height for widget areas
display(HTML("""
<style>
.output_scroll {
    overflow: visible !important;
    height: auto !important;
    max-height: none !important;
}
</style>
"""))


One last thing to note is that we assigned the [`KymoWidgetGreedy`](https://lumicks-pylake.readthedocs.io/en/stable/examples/cas9_kymotracking/../../_api/lumicks.pylake.KymoWidgetGreedy.html#lumicks.pylake.KymoWidgetGreedy) to the variable `kymowidget`. That means that from this point on, we can interact with it through the handle name `kymowidget`.

Exporting from the widget results in a file that contains the track coordinates in pixels and real units. If we also want to export the photon counts in a region around the track, we can include a `sampling_width`. This sums the photon counts from `pixel_position - sampling_width` to (and including) `pixel_position + sampling_width`:

# Save tracks

In [ ]:

# Create a new file name for saving
base_name = os.path.splitext(file_name)[0]  # Remove the .h5 extension
save_file_path = os.path.join(folder_path, f"{base_name}_tracks.txt")  # New file path

# Save the tracks
kymowidget.save_tracks(save_file_path, sampling_width=1)

print(f"Tracks saved to: {save_file_path}")




# Calculate diffusion and Export results in a new folder

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
import csv

PIXEL_TIME_RESOLUTION = 0.3  # seconds per pixel

def compute_msd(position, dt):
    """Compute Mean Squared Displacement given position and time interval (dt)."""
    n = len(position)
    max_lag = n // 2  # Maximum lag to consider
    msd = []

    for lag in range(1, max_lag):
        displacements = position[lag:] - position[:-lag]
        squared_displacement = displacements ** 2
        msd.append(np.mean(squared_displacement))

    lag_times = np.arange(1, max_lag) * dt  # Lag times in seconds
    return lag_times, np.array(msd)

diffusion_results = []
all_msd_data = []

plt.figure(figsize=(10, 6))

for i, track in enumerate(kymowidget.tracks):
    try:
        # Extract position and calculate time points based on 0.3 second resolution
        position = np.array(track.position)  # positions in micrometers
        time = np.arange(len(position)) * PIXEL_TIME_RESOLUTION  # time in seconds, with correct scaling

        if len(position) < 3:
            print(f"Skipping track {i}: too short")
            continue

        # Compute MSD
        lags, msd = compute_msd(position, PIXEL_TIME_RESOLUTION)

        # Linear regression on first few points (assume linear regime)
        num_points = min(5, len(msd))
        slope, intercept, r_value, _, _ = linregress(lags[:num_points], msd[:num_points])
        diffusion_coeff = slope / 2  # for 1D diffusion

        diffusion_results.append((i, diffusion_coeff))

        # Plot MSD
        plt.plot(lags, msd, label=f'Track {i} (D={diffusion_coeff:.3f} µm²/s)')
        
        # Store MSD data for CSV export
        for lag, m in zip(lags, msd):
            all_msd_data.append([i, lag, m])

    except AttributeError:
        print(f"Skipping track {i}: missing position data")

plt.xlabel("Lag Time (s)")
plt.ylabel("MSD (µm²)")
plt.title("Mean Squared Displacement per Track")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Assuming you have a variable `file_name` that holds the kymograph file name
base_name = os.path.splitext(file_name)[0]  # Remove extension from kymograph name

# Define the directory for saving the files
save_directory = os.path.join(os.getcwd(), "directory")  # Working directory + subfolder

# Create the directory if it doesn't exist
os.makedirs(save_directory, exist_ok=True)

# Define the file paths based on the kymograph file name
msd_file_path = os.path.join(save_directory, f"{base_name}_msd.csv")
diffusion_file_path = os.path.join(save_directory, f"{base_name}_dc.csv")

# Save MSD data to CSV
with open(msd_file_path, mode='w', newline='') as msd_file:
    writer = csv.writer(msd_file)
    writer.writerow(["Track Index", "Lag Time (s)", "MSD (µm²)"])  # Write header
    writer.writerows(all_msd_data)

print(f"MSD data saved to {msd_file_path}")

# Save diffusion coefficients to CSV
with open(diffusion_file_path, mode='w', newline='') as diff_file:
    writer = csv.writer(diff_file)
    writer.writerow(["Track Index", "Diffusion Coefficient (µm²/s)"])  # Write header
    writer.writerows(diffusion_results)

print(f"Diffusion coefficients saved to {diffusion_file_path}")

# Print summary
if diffusion_results:
    print("\nEstimated Diffusion Coefficients:")
    for idx, D in diffusion_results:
        print(f"Track {idx}: D = {D:.4f} µm²/s")
else:
    print("No valid tracks to calculate diffusion.")

# COMBINE DIFFUSION FROM ALL MSD

In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt

# Path to the folder containing the CSV files
folder_path = os.path.join(os.getcwd(), "directory")

# Find all MSD files
msd_files = glob.glob(os.path.join(folder_path, "*_msd.csv"))

plt.figure(figsize=(10, 10))

# Loop through each file and plot the data
for msd_file in msd_files:
    df = pd.read_csv(msd_file)
    
    # Plot each track without labels
    for _, group in df.groupby("Track Index"):
        plt.plot(group["Lag Time (s)"], group["MSD (µm²)"], alpha=0.5)

plt.xlabel("Lag Time (s)")
plt.ylabel("MSD (µm²)")
plt.title("MSD Curves from All Kymographs")
plt.ylim(0, 10)  # Set Y-axis range
plt.grid(True)
plt.tight_layout()
plt.show()

# COMPARISON BETWEEN CONDITIONS

In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt

# Define folders and their corresponding colors
folders = {
    "directory/condition_1": "blue",
    "directory/condition_2": "red"
}

plt.figure(figsize=(10, 6))

for folder_name, color in folders.items():
    folder_path = os.path.join(os.getcwd(), folder_name)
    msd_files = glob.glob(os.path.join(folder_path, "*_msd.csv"))

    for msd_file in msd_files:
        df = pd.read_csv(msd_file)

        # Plot each track in the assigned color
        for _, group in df.groupby("Track Index"):
            plt.plot(group["Lag Time (s)"], group["MSD (µm²)"], alpha=0.4, color=color)

# Customize plot
plt.xlabel("Lag Time (s)")
plt.ylabel("MSD (µm²)")
plt.title("MSD Curves from All Kymographs")
plt.ylim(0, 6)
plt.grid(True)
plt.tight_layout()

# Add custom legend
from matplotlib.lines import Line2D
custom_lines = [
    Line2D([0], [0], color='blue', lw=2, label='condition_1'),
    Line2D([0], [0], color='red', lw=2, label='condition_2')
]
plt.legend(handles=custom_lines)

plt.show()

# EXPORT MSD TRACKS FROM  MULTIPLE FILES

In [ ]:
import os
import glob
import pandas as pd

# Define folders and their condition labels
folders = {
    "condition_1": "condition_2"
}

combined_data = []

for folder_name, condition in folders.items():
    folder_path = os.path.join(os.getcwd(), folder_name)
    msd_files = glob.glob(os.path.join(folder_path, "*_msd.csv"))

    for msd_file in msd_files:
        df = pd.read_csv(msd_file)
        df["Condition"] = condition
        combined_data.append(df)

# Concatenate all into one DataFrame
combined_df = pd.concat(combined_data, ignore_index=True)

# Save to CSV for Prism
output_path = os.path.join(os.getcwd(), "combined_msd_data.csv")
combined_df.to_csv(output_path, index=False)

print(f"Combined MSD data saved to: {output_path}")

# EXPORT MSD TRACKS FROM  MULTIPLE FILES AND DIVIDE IN COLUMNS FOR EASIER VISUALIZATION IN PRISM

In [ ]:
import os
import glob
import pandas as pd

# Define folders and their condition labels
folders = folders = {"condition_1": "condition_2"}
all_tracks = []
lag_times = None  # We will store lag time separately

for folder_path, label in folders.items():
    folder_full_path = os.path.join(os.getcwd(), folder_path)
    msd_files = glob.glob(os.path.join(folder_full_path, "*_msd.csv"))

    for file_idx, msd_file in enumerate(msd_files):
        df = pd.read_csv(msd_file)
        track_ids = df["Track Index"].unique()

        for track_id in track_ids:
            track_df = df[df["Track Index"] == track_id][["Lag Time (s)", "MSD (µm²)"]].reset_index(drop=True)
            col_name = f"{label}_{file_idx}_{track_id}"

            if lag_times is None:
                lag_times = track_df["Lag Time (s)"].copy()

            track_df = track_df.rename(columns={"MSD (µm²)": col_name})
            all_tracks.append(track_df[[col_name]])

# Concatenate all columns with lag time at the front
combined_df = pd.concat([lag_times] + all_tracks, axis=1)
combined_df = combined_df.rename(columns={"Lag Time (s)": "Lag Time (s)"})

# Export
output_path = os.path.join(os.getcwd(), "msd_ATP_curves_individual_columns.csv")
combined_df.to_csv(output_path, index=False)
print(f"Exported MSD data with separate columns to: {output_path}")

# Calculate Diffusion constant 

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.linalg import lstsq

# ----------------------------
# Illustrator-friendly settings
# ----------------------------
mpl.rcParams['pdf.fonttype'] = 42     # keep text as TrueType in PDF
mpl.rcParams['ps.fonttype']  = 42
mpl.rcParams['svg.fonttype'] = 'none' # keep text as text in SVG (not paths)

# ----------------------------
# SETTINGS
# ----------------------------
# Your exact RGB colors (normalized 0–1 for matplotlib)
def rgb(r,g,b): return (r/255.0, g/255.0, b/255.0)
palette = {
    "condition_1": rgb(129,206,109),
    "condition_2": rgb(39,79,204),
    "condition_3": rgb(254,204,102),
    "condition_4": rgb(251,2,255),
}

folders = dict(palette)  # same keys define the folders to read

min_points_total = 3        # require at least this many total lag-time points in a track
min_track_duration = 1.0    # require track duration (max lag time) >= 1 s
fit_min, fit_max = 4, 6     # use first 4–6 points for the fit
exclude_negative = True     # drop negative diffusion values
divide_by_2 = True          # convert slope -> physical 1D D = slope/2

save_csv = "diffusion_results_first4to6_all_conditions.csv"  # set None to skip saving
export_base = "Diffusion_by_condition_first4to6_1D"          # base filename for figure export

# ----------------------------
# OLS with intercept; returns slope and sqrt(RSS)
# ----------------------------
def ols_slope_with_intercept(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    X = np.vstack([x, np.ones_like(x)]).T  # design matrix with intercept
    params, residuals, rank, s = lstsq(X, y)
    slope = float(params[0])
    y_fit = X @ params
    rss = float(((y - y_fit) ** 2).sum())
    return slope, np.sqrt(rss)

# ----------------------------
# Collect results
# ----------------------------
results = []

for folder_name, color in folders.items():
    folder_path = os.path.join(os.getcwd(), folder_name)
    msd_files = [f for f in glob.glob(os.path.join(folder_path, "*.csv")) if f.endswith("_msd.csv")]

    if not msd_files:
        print(f"[WARN] No *_msd.csv files found in: {folder_path}")
        continue

    for msd_file in msd_files:
        df = pd.read_csv(msd_file)

        required_cols = {"Track Index", "Lag Time (s)", "MSD (µm²)"}
        if not required_cols.issubset(df.columns):
            print(f"[WARN] Missing required columns in {msd_file}. Skipping.")
            continue

        for track_id, group in df.groupby("Track Index"):
            g = group[["Lag Time (s)", "MSD (µm²)"]].dropna().copy()
            g = g[g["Lag Time (s)"] > 0].sort_values("Lag Time (s)")

            # Basic filters
            if len(g) < min_points_total:
                continue
            if g["Lag Time (s)"].max() < min_track_duration:
                continue

            tau_all = g["Lag Time (s)"].to_numpy(float)
            msd_all = g["MSD (µm²)"].to_numpy(float)

            # Fit on the first k points (k between 4 and 6)
            if len(tau_all) < fit_min:
                continue
            k = min(fit_max, len(tau_all))
            tau_fit = tau_all[:k]
            msd_fit = msd_all[:k]

            slope, rss_sqrt = ols_slope_with_intercept(tau_fit, msd_fit)

            # 1D physical diffusion coefficient
            D_val = slope / 2.0 if divide_by_2 else slope

            # Exclude negative/non-finite
            if exclude_negative and (not np.isfinite(D_val) or D_val < 0):
                continue

            results.append({
                "Condition": folder_name,
                "Track Index": track_id,
                "D": D_val,
                "k_fit_points": k,
            })

# ----------------------------
# Save table & print summary
# ----------------------------
if len(results) == 0:
    raise SystemExit("No valid tracks after filtering. Check folders and CSV contents.")

results_df = pd.DataFrame(results)
if save_csv:
    results_df.to_csv(save_csv, index=False)
    print(f"Saved per-track diffusion results to: {save_csv}")

# Per-condition stats (printed)
summary = []
for cond, sub in results_df.groupby("Condition"):
    D_vals = sub["D"].to_numpy(float)
    D_vals = D_vals[np.isfinite(D_vals)]
    N = D_vals.size
    mean = np.mean(D_vals) if N > 0 else np.nan
    std = np.std(D_vals, ddof=1) if N > 1 else np.nan
    sem = (std / np.sqrt(N)) if (N > 1 and np.isfinite(std)) else np.nan
    summary.append([cond, N, mean, std, sem])

summary_df = pd.DataFrame(summary, columns=["Condition", "N_tracks", "D_mean", "D_std", "D_SEM"])
print("\nAverage Diffusion Constant by Condition (first 4–6 points, negatives excluded, 1D D = slope/2):")
print(summary_df.to_string(index=False))

# ----------------------------
# Plot all conditions together (violin + jitter)
# ----------------------------
sns.set(style="whitegrid")
fig, ax = plt.subplots(figsize=(10, 6))

sns.violinplot(
    data=results_df,
    x="Condition",
    y="D",
    palette=palette,
    inner=None,
    cut=0,
    ax=ax
)

sns.stripplot(
    data=results_df,
    x="Condition",
    y="D",
    palette=palette,
    dodge=False,
    jitter=0.2,
    size=4,
    alpha=0.6,
    ax=ax
)

ax.set_title("Diffusion Coefficient by Condition\n(First 4–6 MSD points, OLS with intercept, 1D: slope/2)")
ax.set_xlabel("Condition")
ax.set_ylabel("D (µm²/s)")
fig.tight_layout()

# ----------------------------
# Illustrator-editable export
# ----------------------------
# Exports as fully-editable vector graphics with your colors and text preserved
fig.savefig(f"{export_base}.svg", bbox_inches='tight')   # SVG (best for Illustrator)
fig.savefig(f"{export_base}.pdf", bbox_inches='tight')   # PDF (also very editable)

plt.show()


In [ ]:
!pip install seaborn

# DWELL TIMES

In [ ]:
import pandas as pd
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Settings
min_dwell_time = 0.5  # seconds
folder = "condition_1"
output_file = f"{folder}_dwell_times.csv"

# Find files
track_files = glob.glob(f"{folder}/*tracks.txt")

# Collect results
all_dwell_data = []

for file_path in track_files:
    try:
        df = pd.read_csv(file_path, sep=';', comment='#', header=None)
        df.columns = [
            'track_id', 'time_px', 'coord_px', 'time_s',
            'position_um', 'counts', 'min_obs_duration'
        ]
        
        dwell_times = df.groupby('track_id')['time_s'].agg(lambda x: x.max() - x.min())
        dwell_times = dwell_times[dwell_times > min_dwell_time]

        file_label = os.path.basename(file_path)
        all_dwell_data.extend([(file_label, tid, dt) for tid, dt in dwell_times.items()])

    except Exception as e:
        print(f"❌ Failed to process {file_path}: {e}")

# Combine results
result_df = pd.DataFrame(all_dwell_data, columns=['file', 'track_id', 'dwell_time_s'])
result_df.to_csv(output_file, index=False)
print(f"✅ Dwell times saved to '{output_file}'")

# --- Plot histogram ---
plt.figure(figsize=(10, 6))
sns.histplot(result_df['dwell_time_s'], bins=30, kde=True, color='skyblue')
plt.xlabel("Dwell Time (s)")
plt.ylabel("Count")
plt.title("Dwell Time Distribution (ATP_lower_C)")
plt.grid(True)
plt.xlim(0, 130)  # <-- fix x-axis limits here
plt.tight_layout()
plt.show()

# --- Statistics ---
overall_stats = result_df['dwell_time_s'].describe()
print("\n📊 Overall Statistics:")
print(overall_stats)

# Per-file stats (mean & count)
file_stats = result_df.groupby('file')['dwell_time_s'].agg(['count', 'mean', 'median', 'std'])
print("\n📁 Per-File Statistics:")
print(file_stats.round(2))
